# ORCID Author Data Query for Wikidata

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

---

## What This Notebook Does

This notebook queries the ORCID Public API to retrieve comprehensive author data for researchers with ORCID identifiers. Starting from a CSV or Excel file containing author names and ORCID IDs (typically exported from CrossRef metadata), it gathers structured biographical and professional information that can later be used to create or update Wikidata items for researchers.

The ORCID registry contains rich metadata about researchers including their names, affiliations, education history, employment history, funding, peer review activities, and works. This notebook extracts this data into a structured format suitable for Wikidata import workflows.

## Key Features

- **ORCID API Integration**: Uses the ORCID Public API v3.0 to retrieve author records
- **Comprehensive Data Extraction**: Gathers name variants, affiliations, education, employment, works, funding, and external identifiers
- **Batch Processing**: Handles multiple ORCID IDs with rate limiting and progress tracking
- **Wikidata-Ready Output**: Structures data to align with Wikidata person item properties
- **Error Handling**: Robust handling of missing data, API errors, and malformed ORCIDs
- **Export Options**: CSV and JSON exports for downstream processing

## Workflow

1. **Setup**: Install dependencies and configure ORCID API credentials (optional for Public API)
2. **Upload Data**: Load CSV/Excel file containing ORCIDs from article metadata
3. **Extract ORCIDs**: Parse and validate ORCID identifiers from input data
4. **Query ORCID API**: Retrieve full author records for each ORCID
5. **Parse Records**: Extract relevant fields for Wikidata mapping
6. **Export Results**: Download structured CSV/JSON for Wikidata import

## Data Fields Retrieved

| ORCID Field | Wikidata Property |
|-------------|------------------|
| ORCID iD | ORCID iD (P496) |
| Given Names | given name (P735) |
| Family Name | family name (P734) |
| Other Names | alias |
| Affiliations | employer (P108), affiliated with (P1416) |
| Education | educated at (P69) |
| Employment | employer (P108) |
| Country | country of citizenship (P27) |
| External IDs | various (Scopus, ResearcherID, etc.) |

## Citation

If you use this notebook, please cite:

> Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

In [ ]:
# Install required packages
!pip install requests pandas openpyxl ipywidgets -q

import requests
import pandas as pd
import time
import re
import json
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("Setup complete.")

## Configuration

The ORCID Public API allows unauthenticated access with rate limits of approximately 24 requests per second. For higher limits or access to member API features, you can register for API credentials.

**Note**: The Public API provides access to all publicly visible data in ORCID records. Private or limited-visibility data requires Member API access with user authorization.

In [ ]:
# Configuration
class Config:
    # ORCID API endpoints
    ORCID_API_BASE = "https://pub.orcid.org/v3.0"
    
    # Rate limiting (Public API allows ~24 req/sec, we'll be conservative)
    REQUEST_DELAY = 0.5  # seconds between requests
    
    # Request settings
    TIMEOUT = 30  # seconds
    MAX_RETRIES = 3
    
    # User-Agent (good practice for API requests)
    USER_AGENT = "ORCID-Wikidata-Query/1.0 (Anthropology Research; mailto:your-email@example.com)"
    
    # Optional: Client credentials for authenticated access (higher rate limits)
    # Leave as None for unauthenticated Public API access
    CLIENT_ID = None
    CLIENT_SECRET = None

config = Config()

print(f"ORCID API Base URL: {config.ORCID_API_BASE}")
print(f"Request delay: {config.REQUEST_DELAY}s")
print(f"Authentication: {'Configured' if config.CLIENT_ID else 'Public API (unauthenticated)'}")

## ORCID Validation and Parsing

ORCID identifiers follow a specific format: `0000-0000-0000-000X` where X can be 0-9 or X (checksum). The functions below handle parsing ORCIDs from various formats found in CrossRef data.

In [ ]:
def validate_orcid(orcid: str) -> bool:
    """
    Validate ORCID checksum using ISO 7064 Mod 11-2 algorithm.
    
    Args:
        orcid: ORCID in format 0000-0000-0000-000X
    
    Returns:
        True if valid, False otherwise
    """
    # Remove hyphens
    digits = orcid.replace('-', '')
    if len(digits) != 16:
        return False
    
    # Compute checksum
    total = 0
    for char in digits[:-1]:
        if not char.isdigit():
            return False
        total = (total + int(char)) * 2
    
    remainder = total % 11
    result = (12 - remainder) % 11
    check_digit = 'X' if result == 10 else str(result)
    
    return digits[-1].upper() == check_digit


def extract_orcid(value: str) -> Optional[str]:
    """
    Extract and normalize ORCID from various input formats.
    
    Handles:
    - Plain ORCID: 0000-0001-2345-6789
    - URL: https://orcid.org/0000-0001-2345-6789
    - URI: orcid:0000-0001-2345-6789
    - No hyphens: 0000000123456789
    
    Args:
        value: String potentially containing an ORCID
    
    Returns:
        Normalized ORCID (0000-0000-0000-000X) or None if invalid
    """
    if not value or not isinstance(value, str):
        return None
    
    value = str(value).strip()
    
    # Pattern to match ORCID with or without hyphens
    patterns = [
        r'(?:https?://)?(?:www\.)?orcid\.org/(\d{4}-?\d{4}-?\d{4}-?\d{3}[\dXx])',
        r'orcid[:/]?(\d{4}-?\d{4}-?\d{4}-?\d{3}[\dXx])',
        r'\b(\d{4}-\d{4}-\d{4}-\d{3}[\dXx])\b',
        r'\b(\d{16})\b',
        r'\b(\d{15}[Xx])\b'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, value, re.IGNORECASE)
        if match:
            orcid = match.group(1).replace('-', '').upper()
            # Format with hyphens
            formatted = f"{orcid[:4]}-{orcid[4:8]}-{orcid[8:12]}-{orcid[12:16]}"
            if validate_orcid(formatted):
                return formatted
    
    return None


def parse_orcids_from_cell(cell_value: str) -> List[str]:
    """
    Parse multiple ORCIDs from a cell (may be semicolon or comma separated).
    
    Args:
        cell_value: Cell content potentially containing multiple ORCIDs
    
    Returns:
        List of valid, normalized ORCIDs
    """
    if not cell_value or pd.isna(cell_value):
        return []
    
    cell_value = str(cell_value)
    
    # Split on common delimiters
    parts = re.split(r'[;,|\s]+', cell_value)
    
    orcids = []
    for part in parts:
        orcid = extract_orcid(part.strip())
        if orcid and orcid not in orcids:
            orcids.append(orcid)
    
    return orcids


# Test validation
test_cases = [
    "0000-0002-1825-0097",  # Valid (Wikipedia example)
    "https://orcid.org/0000-0002-1825-0097",
    "0000000218250097",
    "invalid-orcid",
    "0000-0000-0000-0000"  # Invalid checksum
]

print("ORCID Validation Tests:")
for tc in test_cases:
    result = extract_orcid(tc)
    print(f"  {tc[:40]:<40} → {result}")

## ORCID API Client

The ORCID Public API returns data in XML or JSON format. We'll use JSON for easier parsing.

In [ ]:
class ORCIDClient:
    """
    Client for querying the ORCID Public API.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.session = requests.Session()
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': config.USER_AGENT
        })
        self._last_request_time = 0
        self._access_token = None
        
    def _get_access_token(self) -> Optional[str]:
        """
        Get OAuth access token if credentials are configured.
        Public API works without authentication, but rate limits are higher with it.
        """
        if not self.config.CLIENT_ID or not self.config.CLIENT_SECRET:
            return None
        
        if self._access_token:
            return self._access_token
        
        try:
            response = self.session.post(
                'https://orcid.org/oauth/token',
                data={
                    'client_id': self.config.CLIENT_ID,
                    'client_secret': self.config.CLIENT_SECRET,
                    'grant_type': 'client_credentials',
                    'scope': '/read-public'
                },
                timeout=self.config.TIMEOUT
            )
            response.raise_for_status()
            self._access_token = response.json()['access_token']
            return self._access_token
        except Exception as e:
            print(f"Warning: Could not get access token: {e}")
            return None
    
    def _rate_limit(self):
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self._last_request_time
        if elapsed < self.config.REQUEST_DELAY:
            time.sleep(self.config.REQUEST_DELAY - elapsed)
        self._last_request_time = time.time()
    
    def _make_request(self, endpoint: str) -> Optional[Dict]:
        """
        Make a request to the ORCID API with retries.
        
        Args:
            endpoint: API endpoint path (e.g., '/0000-0002-1825-0097/record')
        
        Returns:
            JSON response or None if failed
        """
        url = f"{self.config.ORCID_API_BASE}{endpoint}"
        
        headers = {}
        token = self._get_access_token()
        if token:
            headers['Authorization'] = f'Bearer {token}'
        
        for attempt in range(self.config.MAX_RETRIES):
            self._rate_limit()
            try:
                response = self.session.get(
                    url,
                    headers=headers,
                    timeout=self.config.TIMEOUT
                )
                
                if response.status_code == 404:
                    return None  # ORCID not found
                
                response.raise_for_status()
                return response.json()
                
            except requests.exceptions.RequestException as e:
                if attempt < self.config.MAX_RETRIES - 1:
                    wait_time = 2 ** attempt
                    print(f"  Retry {attempt + 1}/{self.config.MAX_RETRIES} after {wait_time}s: {e}")
                    time.sleep(wait_time)
                else:
                    print(f"  Failed after {self.config.MAX_RETRIES} attempts: {e}")
                    return None
        
        return None
    
    def get_record(self, orcid: str) -> Optional[Dict]:
        """
        Get the full ORCID record for a researcher.
        
        Args:
            orcid: ORCID identifier (format: 0000-0000-0000-000X)
        
        Returns:
            Full record dictionary or None if not found
        """
        return self._make_request(f'/{orcid}/record')
    
    def get_person(self, orcid: str) -> Optional[Dict]:
        """Get personal information (name, biography, etc.)"""
        return self._make_request(f'/{orcid}/person')
    
    def get_activities(self, orcid: str) -> Optional[Dict]:
        """Get activities (employments, educations, works, etc.)"""
        return self._make_request(f'/{orcid}/activities')
    
    def get_works(self, orcid: str) -> Optional[Dict]:
        """Get works (publications)."""
        return self._make_request(f'/{orcid}/works')


# Initialize client
client = ORCIDClient(config)
print("ORCID API client initialized.")

## ORCID Record Parsing

The ORCID API returns deeply nested JSON structures. These functions extract the relevant fields into a flat structure suitable for Wikidata mapping.

In [ ]:
def safe_get(data: dict, *keys, default=None):
    """
    Safely navigate nested dictionaries.
    
    Args:
        data: Dictionary to traverse
        *keys: Keys to traverse
        default: Default value if path doesn't exist
    
    Returns:
        Value at path or default
    """
    current = data
    for key in keys:
        if isinstance(current, dict) and key in current:
            current = current[key]
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return default
    return current


def parse_date(date_dict: Optional[Dict]) -> Optional[str]:
    """
    Parse ORCID date structure into ISO format.
    
    Args:
        date_dict: ORCID date dictionary with year, month, day sub-dicts
    
    Returns:
        ISO date string or None
    """
    if not date_dict:
        return None
    
    year = safe_get(date_dict, 'year', 'value')
    month = safe_get(date_dict, 'month', 'value')
    day = safe_get(date_dict, 'day', 'value')
    
    if not year:
        return None
    
    if month and day:
        return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
    elif month:
        return f"{year}-{month.zfill(2)}"
    else:
        return year


def parse_name(person_data: Dict) -> Dict:
    """
    Extract name information from ORCID person data.
    
    Returns:
        Dict with given_name, family_name, credit_name, other_names
    """
    name_data = safe_get(person_data, 'name', default={})
    
    result = {
        'given_name': safe_get(name_data, 'given-names', 'value'),
        'family_name': safe_get(name_data, 'family-name', 'value'),
        'credit_name': safe_get(name_data, 'credit-name', 'value'),
        'other_names': []
    }
    
    # Extract other names / aliases
    other_names_data = safe_get(person_data, 'other-names', 'other-name', default=[])
    if other_names_data:
        for other in other_names_data:
            name = safe_get(other, 'content')
            if name:
                result['other_names'].append(name)
    
    return result


def parse_biography(person_data: Dict) -> Optional[str]:
    """Extract biography from person data."""
    return safe_get(person_data, 'biography', 'content')


def parse_keywords(person_data: Dict) -> List[str]:
    """Extract keywords/research interests."""
    keywords = []
    keyword_data = safe_get(person_data, 'keywords', 'keyword', default=[])
    for kw in keyword_data:
        content = safe_get(kw, 'content')
        if content:
            keywords.append(content)
    return keywords


def parse_countries(person_data: Dict) -> List[str]:
    """Extract country codes from addresses."""
    countries = []
    address_data = safe_get(person_data, 'addresses', 'address', default=[])
    for addr in address_data:
        country = safe_get(addr, 'country', 'value')
        if country and country not in countries:
            countries.append(country)
    return countries


def parse_external_ids(person_data: Dict) -> Dict[str, str]:
    """
    Extract external identifiers (Scopus, ResearcherID, etc.).
    
    Returns:
        Dict mapping identifier type to value
    """
    external_ids = {}
    id_data = safe_get(person_data, 'external-identifiers', 'external-identifier', default=[])
    
    for ext_id in id_data:
        id_type = safe_get(ext_id, 'external-id-type')
        id_value = safe_get(ext_id, 'external-id-value')
        if id_type and id_value:
            external_ids[id_type] = id_value
    
    return external_ids


def parse_emails(person_data: Dict) -> List[str]:
    """Extract public email addresses."""
    emails = []
    email_data = safe_get(person_data, 'emails', 'email', default=[])
    for email in email_data:
        addr = safe_get(email, 'email')
        if addr:
            emails.append(addr)
    return emails


def parse_urls(person_data: Dict) -> List[Dict]:
    """Extract researcher URLs (personal websites, etc.)."""
    urls = []
    url_data = safe_get(person_data, 'researcher-urls', 'researcher-url', default=[])
    for url in url_data:
        urls.append({
            'name': safe_get(url, 'url-name'),
            'url': safe_get(url, 'url', 'value')
        })
    return urls


print("Record parsing functions defined.")

In [ ]:
def parse_affiliation(aff: Dict) -> Dict:
    """
    Parse a single affiliation (employment or education) entry.
    
    Returns:
        Dict with organization, role, department, dates, location
    """
    org = safe_get(aff, 'organization', default={})
    
    return {
        'organization_name': safe_get(org, 'name'),
        'organization_city': safe_get(org, 'address', 'city'),
        'organization_region': safe_get(org, 'address', 'region'),
        'organization_country': safe_get(org, 'address', 'country'),
        'disambiguated_org_id': safe_get(org, 'disambiguated-organization', 'disambiguated-organization-identifier'),
        'disambiguated_org_source': safe_get(org, 'disambiguated-organization', 'disambiguation-source'),
        'role_title': safe_get(aff, 'role-title'),
        'department': safe_get(aff, 'department-name'),
        'start_date': parse_date(safe_get(aff, 'start-date')),
        'end_date': parse_date(safe_get(aff, 'end-date')),
        'url': safe_get(aff, 'url', 'value')
    }


def parse_employments(activities_data: Dict) -> List[Dict]:
    """Extract employment history."""
    employments = []
    groups = safe_get(activities_data, 'employments', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            emp = safe_get(summary, 'employment-summary')
            if emp:
                employments.append(parse_affiliation(emp))
    
    return employments


def parse_educations(activities_data: Dict) -> List[Dict]:
    """Extract education history."""
    educations = []
    groups = safe_get(activities_data, 'educations', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            edu = safe_get(summary, 'education-summary')
            if edu:
                educations.append(parse_affiliation(edu))
    
    return educations


def parse_distinctions(activities_data: Dict) -> List[Dict]:
    """Extract distinctions (awards, honors)."""
    distinctions = []
    groups = safe_get(activities_data, 'distinctions', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            dist = safe_get(summary, 'distinction-summary')
            if dist:
                distinctions.append(parse_affiliation(dist))
    
    return distinctions


def parse_memberships(activities_data: Dict) -> List[Dict]:
    """Extract professional memberships."""
    memberships = []
    groups = safe_get(activities_data, 'memberships', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            mem = safe_get(summary, 'membership-summary')
            if mem:
                memberships.append(parse_affiliation(mem))
    
    return memberships


def parse_qualifications(activities_data: Dict) -> List[Dict]:
    """Extract qualifications (certifications, licenses)."""
    qualifications = []
    groups = safe_get(activities_data, 'qualifications', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            qual = safe_get(summary, 'qualification-summary')
            if qual:
                qualifications.append(parse_affiliation(qual))
    
    return qualifications


def parse_services(activities_data: Dict) -> List[Dict]:
    """Extract service activities (editorial boards, committees)."""
    services = []
    groups = safe_get(activities_data, 'services', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            svc = safe_get(summary, 'service-summary')
            if svc:
                services.append(parse_affiliation(svc))
    
    return services


def parse_invited_positions(activities_data: Dict) -> List[Dict]:
    """Extract invited positions (visiting appointments)."""
    positions = []
    groups = safe_get(activities_data, 'invited-positions', 'affiliation-group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'summaries', default=[])
        for summary in summaries:
            pos = safe_get(summary, 'invited-position-summary')
            if pos:
                positions.append(parse_affiliation(pos))
    
    return positions


def parse_fundings(activities_data: Dict) -> List[Dict]:
    """Extract funding/grants."""
    fundings = []
    groups = safe_get(activities_data, 'fundings', 'group', default=[])
    
    for group in groups:
        summaries = safe_get(group, 'funding-summary', default=[])
        for summary in summaries:
            org = safe_get(summary, 'organization', default={})
            fundings.append({
                'title': safe_get(summary, 'title', 'title', 'value'),
                'type': safe_get(summary, 'type'),
                'organization_name': safe_get(org, 'name'),
                'organization_country': safe_get(org, 'address', 'country'),
                'disambiguated_org_id': safe_get(org, 'disambiguated-organization', 'disambiguated-organization-identifier'),
                'start_date': parse_date(safe_get(summary, 'start-date')),
                'end_date': parse_date(safe_get(summary, 'end-date')),
                'url': safe_get(summary, 'url', 'value')
            })
    
    return fundings


def parse_works_summary(activities_data: Dict) -> Dict:
    """
    Get summary statistics of works (not individual work details).
    
    Returns:
        Dict with total_works and works_by_type counts
    """
    works = safe_get(activities_data, 'works', 'group', default=[])
    
    by_type = {}
    for group in works:
        summaries = safe_get(group, 'work-summary', default=[])
        for summary in summaries:
            work_type = safe_get(summary, 'type', default='unknown')
            by_type[work_type] = by_type.get(work_type, 0) + 1
            break  # Only count first summary per group (they're duplicates)
    
    return {
        'total_works': len(works),
        'works_by_type': by_type
    }


print("Activity parsing functions defined.")

In [ ]:
def parse_full_record(orcid: str, record: Dict) -> Dict:
    """
    Parse a full ORCID record into a structured dictionary.
    
    Args:
        orcid: The ORCID identifier
        record: Full record from ORCID API
    
    Returns:
        Structured dictionary with all extracted data
    """
    person_data = safe_get(record, 'person', default={})
    activities_data = safe_get(record, 'activities-summary', default={})
    
    # Parse name info
    name_info = parse_name(person_data)
    
    # Build full name for convenience
    given = name_info['given_name'] or ''
    family = name_info['family_name'] or ''
    full_name = f"{given} {family}".strip()
    
    # Parse all sections
    employments = parse_employments(activities_data)
    educations = parse_educations(activities_data)
    works_summary = parse_works_summary(activities_data)
    
    # Get current affiliation (most recent employment without end date)
    current_affiliation = None
    for emp in employments:
        if not emp['end_date']:
            current_affiliation = emp['organization_name']
            break
    
    return {
        # Identifiers
        'orcid': orcid,
        'orcid_url': f"https://orcid.org/{orcid}",
        
        # Name
        'given_name': name_info['given_name'],
        'family_name': name_info['family_name'],
        'full_name': full_name,
        'credit_name': name_info['credit_name'],
        'other_names': name_info['other_names'],
        
        # Biography
        'biography': parse_biography(person_data),
        'keywords': parse_keywords(person_data),
        'countries': parse_countries(person_data),
        
        # Contact
        'emails': parse_emails(person_data),
        'urls': parse_urls(person_data),
        
        # External IDs (Scopus, ResearcherID, etc.)
        'external_ids': parse_external_ids(person_data),
        
        # Current affiliation (convenience field)
        'current_affiliation': current_affiliation,
        
        # Affiliations
        'employments': employments,
        'educations': educations,
        'distinctions': parse_distinctions(activities_data),
        'memberships': parse_memberships(activities_data),
        'qualifications': parse_qualifications(activities_data),
        'services': parse_services(activities_data),
        'invited_positions': parse_invited_positions(activities_data),
        
        # Funding
        'fundings': parse_fundings(activities_data),
        
        # Works summary (not individual works)
        'works_count': works_summary['total_works'],
        'works_by_type': works_summary['works_by_type'],
        
        # Metadata
        'retrieved_at': datetime.now().isoformat(),
        'api_version': '3.0'
    }


print("Full record parsing function defined.")

## Upload Input File

Upload your CSV or Excel file containing ORCIDs. The file should have an `ORCIDs` column (as seen in CrossRef exports) or similar column containing ORCID identifiers.

In [ ]:
# Upload file
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Upload your CSV or Excel file containing ORCID data:")
print("(Expected columns include 'ORCIDs', 'Authors', 'Title', 'DOI', etc.)")
print()

if IN_COLAB:
    uploaded = files.upload()
else:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    file_path = filedialog.askopenfilename(
        title="Select CSV or Excel file",
        filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
    )
    if file_path:
        import shutil, os
        filename = os.path.basename(file_path)
        shutil.copy2(file_path, filename)
        uploaded = {filename: open(filename, 'rb').read()}
        print(f"Loaded: {filename}")
    else:
        uploaded = {}
        print("No file selected.")

In [ ]:
# Load uploaded file
filename = list(uploaded.keys())[0]
print(f"Loading: {filename}")

if filename.endswith('.xlsx') or filename.endswith('.xls'):
    df_input = pd.read_excel(filename)
else:
    df_input = pd.read_csv(filename)

print(f"\nLoaded {len(df_input)} rows")
print(f"\nColumns: {df_input.columns.tolist()}")
print(f"\nFirst few rows:")
display(df_input.head())

## Extract Unique ORCIDs

Parse all ORCID identifiers from the input data and deduplicate.

In [ ]:
# Find ORCID column
orcid_column = None
possible_columns = ['ORCIDs', 'ORCID', 'orcid', 'orcids', 'ORCID iD', 'orcid_id', 'author_orcid']

for col in possible_columns:
    if col in df_input.columns:
        orcid_column = col
        break

if not orcid_column:
    print("Available columns:")
    for i, col in enumerate(df_input.columns):
        print(f"  {i}: {col}")
    print("\nEnter the column name containing ORCIDs:")
else:
    print(f"Found ORCID column: '{orcid_column}'")

In [ ]:
# Extract all unique ORCIDs
# Set orcid_column manually if not auto-detected:
# orcid_column = 'YOUR_COLUMN_NAME'

all_orcids = []
orcid_to_rows = {}  # Track which rows each ORCID appears in

for idx, row in df_input.iterrows():
    cell_value = row.get(orcid_column)
    orcids_in_cell = parse_orcids_from_cell(cell_value)
    
    for orcid in orcids_in_cell:
        if orcid not in orcid_to_rows:
            orcid_to_rows[orcid] = []
            all_orcids.append(orcid)
        orcid_to_rows[orcid].append(idx)

print(f"Found {len(all_orcids)} unique ORCIDs")
print(f"\nFirst 10 ORCIDs:")
for orcid in all_orcids[:10]:
    print(f"  {orcid} (appears in {len(orcid_to_rows[orcid])} rows)")

## Query ORCID API

Retrieve full records for all unique ORCIDs. This may take a while for large datasets due to rate limiting.

In [ ]:
# Query ORCID API for all unique ORCIDs

results = []
errors = []

# Progress tracking
progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=len(all_orcids),
    description='Progress:',
    bar_style='info'
)
status_label = widgets.HTML(value='')
display(widgets.VBox([progress_bar, status_label]))

start_time = time.time()

for i, orcid in enumerate(all_orcids):
    progress_bar.value = i + 1
    elapsed = time.time() - start_time
    rate = (i + 1) / elapsed if elapsed > 0 else 0
    remaining = (len(all_orcids) - i - 1) / rate if rate > 0 else 0
    
    status_label.value = f"<b>Querying:</b> {orcid} ({i+1}/{len(all_orcids)}) | Rate: {rate:.1f}/s | Est. remaining: {remaining/60:.1f} min"
    
    try:
        record = client.get_record(orcid)
        
        if record:
            parsed = parse_full_record(orcid, record)
            parsed['source_rows'] = orcid_to_rows[orcid]
            results.append(parsed)
        else:
            errors.append({'orcid': orcid, 'error': 'Not found (404)'})
            
    except Exception as e:
        errors.append({'orcid': orcid, 'error': str(e)})

progress_bar.bar_style = 'success'
status_label.value = f"<b>Complete!</b> Retrieved {len(results)} records, {len(errors)} errors"

print(f"\n\nResults Summary:")
print(f"  Successfully retrieved: {len(results)}")
print(f"  Errors: {len(errors)}")

In [ ]:
# Show any errors
if errors:
    print("Errors encountered:")
    for err in errors[:20]:
        print(f"  {err['orcid']}: {err['error']}")
    if len(errors) > 20:
        print(f"  ... and {len(errors) - 20} more")

## Preview Results

In [ ]:
# Preview first result
if results:
    print("Sample record:")
    sample = results[0]
    print(f"\nORCID: {sample['orcid']}")
    print(f"Name: {sample['full_name']}")
    print(f"Credit Name: {sample['credit_name']}")
    print(f"Other Names: {sample['other_names']}")
    print(f"Current Affiliation: {sample['current_affiliation']}")
    print(f"Countries: {sample['countries']}")
    print(f"Keywords: {sample['keywords']}")
    print(f"Works Count: {sample['works_count']}")
    print(f"External IDs: {sample['external_ids']}")
    
    if sample['employments']:
        print(f"\nEmployments ({len(sample['employments'])}):")
        for emp in sample['employments'][:3]:
            print(f"  - {emp['role_title']} at {emp['organization_name']} ({emp['start_date']} - {emp['end_date'] or 'present'})")
    
    if sample['educations']:
        print(f"\nEducations ({len(sample['educations'])}):")
        for edu in sample['educations'][:3]:
            print(f"  - {edu['role_title']} at {edu['organization_name']} ({edu['start_date']} - {edu['end_date'] or 'present'})")

## Export Results

Export the retrieved data in multiple formats:
1. **Flat CSV** - One row per author with key fields
2. **Full JSON** - Complete nested data for all records
3. **Affiliations CSV** - Separate file with employment/education details

In [ ]:
# Create flat DataFrame for main export
flat_records = []

for r in results:
    flat_records.append({
        'orcid': r['orcid'],
        'orcid_url': r['orcid_url'],
        'given_name': r['given_name'],
        'family_name': r['family_name'],
        'full_name': r['full_name'],
        'credit_name': r['credit_name'],
        'other_names': '; '.join(r['other_names']) if r['other_names'] else '',
        'current_affiliation': r['current_affiliation'],
        'countries': '; '.join(r['countries']) if r['countries'] else '',
        'keywords': '; '.join(r['keywords']) if r['keywords'] else '',
        'biography': r['biography'],
        'emails': '; '.join(r['emails']) if r['emails'] else '',
        'urls': '; '.join([u['url'] for u in r['urls'] if u['url']]) if r['urls'] else '',
        'works_count': r['works_count'],
        'employment_count': len(r['employments']),
        'education_count': len(r['educations']),
        'funding_count': len(r['fundings']),
        # External identifiers
        'scopus_id': r['external_ids'].get('Scopus Author ID'),
        'researcher_id': r['external_ids'].get('ResearcherID'),
        'loop_profile': r['external_ids'].get('Loop profile'),
        'isni': r['external_ids'].get('ISNI'),
        'retrieved_at': r['retrieved_at']
    })

df_results = pd.DataFrame(flat_records)
print(f"Created flat DataFrame with {len(df_results)} records")
display(df_results.head())

In [ ]:
# Create affiliations DataFrame (employments + educations)
affiliation_records = []

for r in results:
    for emp in r['employments']:
        affiliation_records.append({
            'orcid': r['orcid'],
            'full_name': r['full_name'],
            'affiliation_type': 'employment',
            'organization_name': emp['organization_name'],
            'organization_city': emp['organization_city'],
            'organization_region': emp['organization_region'],
            'organization_country': emp['organization_country'],
            'disambiguated_org_id': emp['disambiguated_org_id'],
            'disambiguated_org_source': emp['disambiguated_org_source'],
            'role_title': emp['role_title'],
            'department': emp['department'],
            'start_date': emp['start_date'],
            'end_date': emp['end_date'],
            'is_current': emp['end_date'] is None
        })
    
    for edu in r['educations']:
        affiliation_records.append({
            'orcid': r['orcid'],
            'full_name': r['full_name'],
            'affiliation_type': 'education',
            'organization_name': edu['organization_name'],
            'organization_city': edu['organization_city'],
            'organization_region': edu['organization_region'],
            'organization_country': edu['organization_country'],
            'disambiguated_org_id': edu['disambiguated_org_id'],
            'disambiguated_org_source': edu['disambiguated_org_source'],
            'role_title': edu['role_title'],  # Degree type
            'department': edu['department'],
            'start_date': edu['start_date'],
            'end_date': edu['end_date'],
            'is_current': edu['end_date'] is None
        })

df_affiliations = pd.DataFrame(affiliation_records)
print(f"Created affiliations DataFrame with {len(df_affiliations)} records")
if len(df_affiliations) > 0:
    display(df_affiliations.head())

In [ ]:
# Export files
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Main CSV
csv_filename = f"orcid_author_data_{timestamp}.csv"
df_results.to_csv(csv_filename, index=False)
print(f"Saved: {csv_filename}")

# Affiliations CSV
if len(df_affiliations) > 0:
    aff_filename = f"orcid_affiliations_{timestamp}.csv"
    df_affiliations.to_csv(aff_filename, index=False)
    print(f"Saved: {aff_filename}")

# Full JSON (with all nested data)
json_filename = f"orcid_full_records_{timestamp}.json"
with open(json_filename, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Saved: {json_filename}")

# Errors log
if errors:
    errors_filename = f"orcid_errors_{timestamp}.csv"
    pd.DataFrame(errors).to_csv(errors_filename, index=False)
    print(f"Saved: {errors_filename}")

In [ ]:
# Download files
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Downloading files...")

if IN_COLAB:
    files.download(csv_filename)

    if len(df_affiliations) > 0:
        files.download(aff_filename)

    files.download(json_filename)

    if errors:
        files.download(errors_filename)

print("\nDownload complete!")
if not IN_COLAB:
    print(f"Files saved to working directory.")

## Summary Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("ORCID DATA RETRIEVAL SUMMARY")
print("=" * 60)

print(f"\nInput:")
print(f"  Rows in input file: {len(df_input)}")
print(f"  Unique ORCIDs found: {len(all_orcids)}")

print(f"\nResults:")
print(f"  Records retrieved: {len(results)}")
print(f"  Errors: {len(errors)}")

if results:
    # Counts
    with_affiliation = sum(1 for r in results if r['current_affiliation'])
    with_education = sum(1 for r in results if r['educations'])
    with_works = sum(1 for r in results if r['works_count'] > 0)
    with_external_ids = sum(1 for r in results if r['external_ids'])
    
    print(f"\nData Completeness:")
    print(f"  With current affiliation: {with_affiliation} ({100*with_affiliation/len(results):.1f}%)")
    print(f"  With education history: {with_education} ({100*with_education/len(results):.1f}%)")
    print(f"  With works listed: {with_works} ({100*with_works/len(results):.1f}%)")
    print(f"  With external identifiers: {with_external_ids} ({100*with_external_ids/len(results):.1f}%)")
    
    # Works statistics
    total_works = sum(r['works_count'] for r in results)
    avg_works = total_works / len(results) if results else 0
    print(f"\nWorks:")
    print(f"  Total works across all authors: {total_works}")
    print(f"  Average works per author: {avg_works:.1f}")
    
    # Countries
    all_countries = []
    for r in results:
        all_countries.extend(r['countries'])
    if all_countries:
        country_counts = pd.Series(all_countries).value_counts().head(10)
        print(f"\nTop Countries:")
        for country, count in country_counts.items():
            print(f"  {country}: {count}")

print("\n" + "=" * 60)

## Wikidata Property Mapping Reference

Use this reference when creating Wikidata items from the exported data:

| ORCID Field | Wikidata Property | Notes |
|-------------|-------------------|-------|
| `orcid` | ORCID iD (P496) | Format without URL prefix |
| `given_name` | given name (P735) | Link to name item if exists |
| `family_name` | family name (P734) | Link to name item if exists |
| `other_names` | alias | Add as aliases on item |
| `current_affiliation` | employer (P108) | Link to organization QID |
| `educations` | educated at (P69) | With qualifiers for dates, degree |
| `employments` | employer (P108) | With start/end time qualifiers |
| `countries` | country of citizenship (P27) | If appropriate |
| `scopus_id` | Scopus author ID (P1153) | |
| `researcher_id` | ResearcherID (P1053) | |
| `isni` | ISNI (P213) | |
| `disambiguated_org_id` | various | GRID→P2427, ROR→P6782, Ringgold→P3500 |